# V3 Feature-Batch Evaluation

This notebook compares the different observation feature batches for one fixed RL experiment setting.

What it does:
- filters the sweep CSV to one algorithm/reward/weight setting
- averages each feature batch across all tested seeds
- compares feature batches on runtime, 2Q gate reduction, and depth reduction
- shows both raw metric plots and normalized comparison views
- optionally saves clean figure files for reports


## 1. Config

Keep the experiment settings here so the rest of the notebook stays simple.


In [ ]:
from pathlib import Path
import csv
import json

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="ticks", context="paper")
plt.rcParams["figure.dpi"] = 130

# Publication figure colors. Set both background colors to '#FFFFFF' for white figures.
FIGURE_BACKGROUND_COLOR = "#FFFFFF"
AXES_BACKGROUND_COLOR = "#FFFFFF"
FIGURE_TEXT_COLOR = "#24333D"
FIGURE_BORDER_COLOR = "#657681"
FIGURE_GRID_COLOR = "#D4DCE0"
BAR_EDGE_COLOR = "#40515F"

# Raw three-panel metric comparison: feature_batch_*_raw_metric_comparison.
# Main title above all three raw-metric panels.
RAW_MAIN_TITLE_SIZE = 16
# Runtime, 2Q reduction, and depth-reduction titles above individual panels.
RAW_PANEL_TITLE_SIZE = 13
# Feature-batch names along the y-axis of the raw-metric panels.
RAW_BATCH_NAME_SIZE = 10
# Numbered x-axis metric ticks in the raw-metric panels.
RAW_X_TICK_SIZE = 9
# Metric values printed at the end of raw-metric bars.
RAW_BAR_VALUE_SIZE = 9
# Optional reference-line legend beneath the raw-metric figure.
RAW_LEGEND_SIZE = 9

# Normalized feature-batch heatmap: feature_batch_*_heatmap.
# Main heatmap title.
HEATMAP_TITLE_SIZE = 14
# Feature-batch names along the heatmap y-axis.
HEATMAP_Y_TICK_SIZE = 10
# Metric names along the heatmap x-axis.
HEATMAP_X_TICK_SIZE = 10
# Normalized values printed inside heatmap cells.
HEATMAP_CELL_VALUE_SIZE = 9
# Numbered tick values beside the heatmap colorbar.
HEATMAP_COLORBAR_TICK_SIZE = 9
# 'Normalized score' label beside the heatmap colorbar.
HEATMAP_COLORBAR_LABEL_SIZE = 11

# Three-panel delta figure: feature_batch_delta_vs_original_no_category_*.
# Main title above all three delta panels.
DELTA_MAIN_TITLE_SIZE = 15
# Metric title above each delta panel.
DELTA_PANEL_TITLE_SIZE = 12
# Feature-batch names along the delta-panel y-axes.
DELTA_BATCH_NAME_SIZE = 9
# Numbered x-axis delta ticks in each panel.
DELTA_X_TICK_SIZE = 9
# Delta values printed at the end of bars.
DELTA_BAR_VALUE_SIZE = 9


def style_evaluation_axes(ax, grid_axis="x"):
    """Apply the shared enclosed, scientific-paper figure style."""
    ax.figure.set_facecolor(FIGURE_BACKGROUND_COLOR)
    ax.set_facecolor(AXES_BACKGROUND_COLOR)
    ax.set_axisbelow(True)
    ax.tick_params(axis="both", colors=FIGURE_TEXT_COLOR, width=0.8)
    ax.xaxis.label.set_color(FIGURE_TEXT_COLOR)
    ax.yaxis.label.set_color(FIGURE_TEXT_COLOR)
    ax.title.set_color(FIGURE_TEXT_COLOR)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(FIGURE_BORDER_COLOR)
        spine.set_linewidth(0.8)
    ax.grid(False)
    if grid_axis == "x":
        ax.xaxis.grid(True, color=FIGURE_GRID_COLOR, linewidth=0.65, alpha=0.8)
    elif grid_axis == "y":
        ax.yaxis.grid(True, color=FIGURE_GRID_COLOR, linewidth=0.65, alpha=0.8)

RESULTS_FILE_CANDIDATES = [
    "grid_search_runs_feature_batches.csv",
    "grid_search_runs.csv",
    "grid_search_runs_original_features.csv",
]

FEATURE_BATCH_ORDER = [
    "original",
    "original_no_category",
    "plus_gate_types",
    "plus_ratios",
    "plus_interaction",
    "plus_dynamic_progress",
    "plus_gate_types_ratios",
    "plus_all_static",
    "plus_all_features",
]

FEATURE_BATCH_LABELS = {
    "original": "Original + category",
    "original_no_category": "Original",
    "plus_gate_types": "+ gate types",
    "plus_ratios": "+ ratios",
    "plus_interaction": "+ interaction",
    "plus_dynamic_progress": "+ dynamic progress",
    "plus_gate_types_ratios": "+ gate types + ratios",
    "plus_all_static": "+ all static",
    "plus_all_features": "+ all features",
}

FILTERS = {
    "algorithm_name": "ppo",
    "reward_type": "mixed_weighted",
    "runtime_weight": 0.1,
    "two_qubit_weight": 1.0,
    "depth_weight": 0.1,
    "final_reward_weight": 0.5,
}

# Easy switch: choose which split columns are emphasized in summary and plots.
# Options: "validation" or "test"
EVAL_SPLIT = "validation"
if EVAL_SPLIT not in {"validation", "test"}:
    raise ValueError("EVAL_SPLIT must be 'validation' or 'test'.")

# Optional figure export.
SAVE_FIGURES = True
FIGURE_DIR_NAME = "feature_batch_evaluation_figures"

DEFAULT_FEATURE_BATCH = "original"
SELECTED_UTILITY_COL = f"{EVAL_SPLIT}_utility"
SELECTED_2Q_COL = f"{EVAL_SPLIT}_avg_two_qubit_gate_reduction_pct"
SELECTED_DEPTH_COL = f"{EVAL_SPLIT}_avg_depth_reduction_pct"
SELECTED_RUNTIME_COL = f"{EVAL_SPLIT}_avg_runtime_seconds"
SELECTED_RUNTIME_SCORE_COL = f"{EVAL_SPLIT}_avg_runtime_score"

CURRENT_SWEEP_COLUMNS = [
    "sweep_id",
    "run_index",
    "total_runs",
    "status",
    "algorithm_name",
    "algorithm_seed",
    "feature_batch",
    "reward_type",
    "runtime_weight",
    "two_qubit_weight",
    "depth_weight",
    "final_reward_weight",
    "total_timesteps",
    "grid_search_timesteps",
    "grid_search_candidates",
    "elapsed_seconds",
    "model_path",
    "best_config",
    "validation_utility",
    "test_utility",
    "validation_avg_two_qubit_gate_reduction_pct",
    "test_avg_two_qubit_gate_reduction_pct",
    "validation_avg_depth_reduction_pct",
    "test_avg_depth_reduction_pct",
    "validation_avg_runtime_seconds",
    "test_avg_runtime_seconds",
    "validation_avg_runtime_score",
    "test_avg_runtime_score",
    "error_message",
]

NUMERIC_COLUMNS = [
    "runtime_weight",
    "two_qubit_weight",
    "depth_weight",
    "final_reward_weight",
    "validation_utility",
    "test_utility",
    "validation_avg_two_qubit_gate_reduction_pct",
    "test_avg_two_qubit_gate_reduction_pct",
    "validation_avg_depth_reduction_pct",
    "test_avg_depth_reduction_pct",
    "validation_avg_runtime_seconds",
    "test_avg_runtime_seconds",
    "validation_avg_runtime_score",
    "test_avg_runtime_score",
    "elapsed_seconds",
]


def resolve_notebook_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "machine_learning" / "RL" / "two_opt_chain_rl_v3",
    ]
    for candidate in candidates:
        if not candidate.is_dir():
            continue
        if any((candidate / name).exists() for name in RESULTS_FILE_CANDIDATES):
            return candidate
    raise FileNotFoundError(
        "Could not locate the v3 feature-batch notebook directory."
    )


NOTEBOOK_DIR = resolve_notebook_dir()
RESULTS_CSV = next(
    NOTEBOOK_DIR / name
    for name in RESULTS_FILE_CANDIDATES
    if (NOTEBOOK_DIR / name).exists()
)
FIGURE_DIR = NOTEBOOK_DIR / FIGURE_DIR_NAME
if SAVE_FIGURES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

summary = pd.DataFrame([{
    "results_csv": RESULTS_CSV.name,
    "evaluation_split": EVAL_SPLIT,
    **FILTERS,
}])
display(summary)


## 2. Load, Filter, And Average Across Seeds

This section keeps only successful runs that match the exact experiment setting, averages duplicate runs within each `feature_batch + seed`, then averages those seed-level results into one summary row per feature batch.


In [ ]:
def load_current_sweep_csv(csv_path):
    rows = []
    with open(csv_path, newline="") as handle:
        reader = csv.reader(handle)
        header = next(reader, None)
        for row in reader:
            if not row:
                continue
            if len(row) != len(CURRENT_SWEEP_COLUMNS):
                continue
            parsed = dict(zip(CURRENT_SWEEP_COLUMNS, row))
            parsed.setdefault("algorithm_seed", 42)
            if parsed["algorithm_seed"] in {None, ""}:
                parsed["algorithm_seed"] = 42
            parsed.setdefault("feature_batch", DEFAULT_FEATURE_BATCH)
            if parsed["feature_batch"] in {None, ""}:
                parsed["feature_batch"] = DEFAULT_FEATURE_BATCH
            rows.append(parsed)
    return pd.DataFrame(rows)


results = load_current_sweep_csv(RESULTS_CSV)
if results.empty:
    raise ValueError(f"No rows were loaded from {RESULTS_CSV}.")

for column in NUMERIC_COLUMNS:
    if column in results.columns:
        results[column] = pd.to_numeric(results[column], errors="coerce")

results["algorithm_seed"] = pd.to_numeric(results["algorithm_seed"], errors="coerce").fillna(42).astype(int)
results["feature_batch"] = results["feature_batch"].fillna(DEFAULT_FEATURE_BATCH)
results = results[results["status"] == "success"].copy()

for key, value in FILTERS.items():
    if isinstance(value, float):
        results = results[np.isclose(results[key].astype(float), value)].copy()
    else:
        results = results[results[key] == value].copy()

results = results[results["feature_batch"] != "original"].copy()
required_columns = [
    "feature_batch",
    "algorithm_seed",
    SELECTED_UTILITY_COL,
    SELECTED_2Q_COL,
    SELECTED_DEPTH_COL,
    SELECTED_RUNTIME_COL,
]
missing = [column for column in required_columns if column not in results.columns]
if missing:
    raise ValueError(f"Missing required result columns: {missing}")

results = results.dropna(subset=required_columns).copy()
if results.empty:
    raise ValueError(
        "No successful runs match the selected algorithm/reward/weight setting."
    )

results["feature_batch_label"] = results["feature_batch"].map(FEATURE_BATCH_LABELS).fillna(results["feature_batch"])
results["feature_batch_sort"] = results["feature_batch"].apply(
    lambda name: FEATURE_BATCH_ORDER.index(name) if name in FEATURE_BATCH_ORDER else 999
)
results["run_index"] = pd.to_numeric(results["run_index"], errors="coerce")
results["sweep_id"] = results["sweep_id"].fillna("")

metric_columns = [
    "validation_utility",
    "test_utility",
    "validation_avg_two_qubit_gate_reduction_pct",
    "test_avg_two_qubit_gate_reduction_pct",
    "validation_avg_depth_reduction_pct",
    "test_avg_depth_reduction_pct",
    "validation_avg_runtime_seconds",
    "test_avg_runtime_seconds",
    "validation_avg_runtime_score",
    "test_avg_runtime_score",
    "elapsed_seconds",
]

seed_level = (
    results.groupby(["feature_batch", "algorithm_seed"], as_index=False)
    .agg(
        algorithm_name=("algorithm_name", "first"),
        reward_type=("reward_type", "first"),
        runtime_weight=("runtime_weight", "first"),
        two_qubit_weight=("two_qubit_weight", "first"),
        depth_weight=("depth_weight", "first"),
        final_reward_weight=("final_reward_weight", "first"),
        feature_batch_sort=("feature_batch_sort", "first"),
        feature_batch_label=("feature_batch_label", "first"),
        successful_runs_for_seed=("feature_batch", "size"),
        **{column: (column, "mean") for column in metric_columns},
    )
)

feature_batch_summary = (
    seed_level.groupby("feature_batch", as_index=False)
    .agg(
        algorithm_name=("algorithm_name", "first"),
        reward_type=("reward_type", "first"),
        feature_batch_sort=("feature_batch_sort", "first"),
        feature_batch_label=("feature_batch_label", "first"),
        tested_seed_count=("algorithm_seed", "nunique"),
        tested_seeds=("algorithm_seed", lambda values: ", ".join(str(int(v)) for v in sorted(pd.unique(values)))),
        successful_runs=("successful_runs_for_seed", "sum"),
        **{column: (column, "mean") for column in metric_columns},
    )
    .sort_values(by=["feature_batch_sort", "feature_batch_label"])
    .reset_index(drop=True)
)

summary = pd.DataFrame([{
    "matching_successful_runs": len(results),
    "seed_level_rows": len(seed_level),
    "feature_batches_found": feature_batch_summary["feature_batch"].nunique(),
    "tested_seeds_found": results["algorithm_seed"].nunique(),
    "results_csv": RESULTS_CSV.name,
}])

run_counts = feature_batch_summary[[
    "feature_batch",
    "feature_batch_label",
    "tested_seed_count",
    "tested_seeds",
    "successful_runs",
]].copy()

comparison = feature_batch_summary.copy()

display(summary)
display(run_counts.sort_values("feature_batch_label"))
display(
    feature_batch_summary[
        [
            "feature_batch_label",
            "tested_seed_count",
            "tested_seeds",
            SELECTED_UTILITY_COL,
            SELECTED_2Q_COL,
            SELECTED_DEPTH_COL,
            SELECTED_RUNTIME_COL,
        ]
    ]
)


## 3. Build A Clean Comparison Table

This section starts from the seed-averaged feature-batch summary and computes ranks plus normalized scores for easier plotting.


In [ ]:
comparison = comparison.copy()
comparison["feature_batch_label"] = comparison["feature_batch_label"].astype(str)

comparison["validation_2q"] = pd.to_numeric(
    comparison["validation_avg_two_qubit_gate_reduction_pct"], errors="coerce"
)
comparison["test_2q"] = pd.to_numeric(
    comparison["test_avg_two_qubit_gate_reduction_pct"], errors="coerce"
)
comparison["validation_depth"] = pd.to_numeric(
    comparison["validation_avg_depth_reduction_pct"], errors="coerce"
)
comparison["test_depth"] = pd.to_numeric(
    comparison["test_avg_depth_reduction_pct"], errors="coerce"
)
comparison["validation_runtime"] = pd.to_numeric(
    comparison["validation_avg_runtime_seconds"], errors="coerce"
)
comparison["test_runtime"] = pd.to_numeric(
    comparison["test_avg_runtime_seconds"], errors="coerce"
)

comparison["validation_2q_rank"] = comparison["validation_2q"].rank(ascending=False, method="min")
comparison["validation_depth_rank"] = comparison["validation_depth"].rank(ascending=False, method="min")
comparison["validation_runtime_rank"] = comparison["validation_runtime"].rank(ascending=True, method="min")


def normalize_higher_better(series):
    series = pd.to_numeric(series, errors="coerce")
    min_value = series.min()
    max_value = series.max()
    if pd.isna(min_value) or pd.isna(max_value):
        return pd.Series(np.nan, index=series.index)
    if min_value == max_value:
        return pd.Series(1.0, index=series.index)
    return (series - min_value) / (max_value - min_value)


def normalize_lower_better(series):
    return 1.0 - normalize_higher_better(series)


comparison["validation_2q_score"] = normalize_higher_better(comparison["validation_2q"])
comparison["validation_depth_score"] = normalize_higher_better(comparison["validation_depth"])
comparison["validation_runtime_score_norm"] = normalize_lower_better(comparison["validation_runtime"])
comparison["validation_composite_score"] = (
    comparison["validation_2q_score"]
    + comparison["validation_depth_score"]
    + comparison["validation_runtime_score_norm"]
) / 3.0

summary_columns = [
    "feature_batch_label",
    "tested_seed_count",
    "tested_seeds",
    "validation_2q",
    "validation_depth",
    "validation_runtime",
    "test_2q",
    "test_depth",
    "test_runtime",
    "validation_2q_rank",
    "validation_depth_rank",
    "validation_runtime_rank",
    "validation_composite_score",
]

display(comparison[summary_columns].sort_values("validation_composite_score", ascending=False))


## 4. Raw Metric Comparison Plots

These plots compare the seed-averaged feature batches directly on runtime, 2Q reduction, and depth reduction for both validation and test.

For the 2Q and depth charts, the notebook overlays labeled reference lines for the best single optimizer and the best fixed two-step chain on the matching split.

For the runtime charts, the notebook overlays the runtime of the best 2Q single optimizer and the best 2Q fixed two-step chain on the matching split.



In [ ]:
PALETTE = sns.color_palette("colorblind", n_colors=max(len(comparison), 3))
BATCH_COLOR_MAP = dict(zip(comparison["feature_batch_label"], PALETTE))

# Manually edit these so each reference line keeps the same appearance everywhere.
REFERENCE_STYLE_MAP = {
    "2q_single": {"color": "red", "linestyle": "--"},
    "2q_chain": {"color": "orange", "linestyle": "-."},
    "depth_single": {"color": "#6baed6", "linestyle": "--"},
    "depth_chain": {"color": "green", "linestyle": "-."},
    "runtime_2q_single": {"color": "red", "linestyle": "--"},
    "runtime_2q_chain": {"color": "orange", "linestyle": "-."},
    "runtime_depth_single": {"color": "#6baed6", "linestyle": "--"},
    "runtime_depth_chain": {"color": "green", "linestyle": "-."},

}


def save_figure(fig, stem):
    if not SAVE_FIGURES:
        return
    fig.savefig(
        FIGURE_DIR / f"{stem}.png",
        bbox_inches="tight",
        dpi=300,
        facecolor=fig.get_facecolor(),
    )
    fig.savefig(
        FIGURE_DIR / f"{stem}.pdf",
        bbox_inches="tight",
        facecolor=fig.get_facecolor(),
    )


def format_chain_label(chain_name):
    return str(chain_name).replace("__", " -> ")


def build_best_by_metric_row(frame, rank_column, value_column, style_key, short_label):
    if frame.empty:
        return None

    ranked = (
        frame.groupby("chain_name", as_index=False)
        .agg(
            rank_value=(rank_column, "mean"),
            display_value=(value_column, "mean"),
        )
        .sort_values("rank_value", ascending=False)
    )
    best_row = ranked.iloc[0]
    return {
        "style_key": style_key,
        "label": f"Best {short_label}: {format_chain_label(best_row['chain_name'])}",
        "value": float(best_row["display_value"]),
    }


def build_oracle_metric_row(frame, circuit_column, value_column, style_key, short_label):
    if frame.empty:
        return None

    oracle_per_circuit = (
        frame.groupby(circuit_column, as_index=False)[value_column]
        .max()
    )
    oracle_value = float(oracle_per_circuit[value_column].mean())
    return {
        "style_key": style_key,
        "label": f"Oracle upper bound {short_label}",
        "value": oracle_value,
    }


def load_split_references(split_name):
    split_csv = NOTEBOOK_DIR / "splits" / f"{split_name}.csv"
    split_rows = pd.read_csv(split_csv)

    required_columns = [
        "chain_name",
        "total_two_qubit_gate_reduction_pct",
        "depth_reduction_pct",
        "total_runtime_seconds",
    ]
    missing_columns = [
        column for column in required_columns if column not in split_rows.columns
    ]
    if missing_columns:
        raise ValueError(
            f"Missing required split columns in {split_csv.name}: {missing_columns}"
        )

    split_rows = split_rows.copy()
    split_rows["chain_name"] = split_rows["chain_name"].astype(str)
    split_rows["total_two_qubit_gate_reduction_pct"] = pd.to_numeric(
        split_rows["total_two_qubit_gate_reduction_pct"], errors="coerce"
    )
    split_rows["depth_reduction_pct"] = pd.to_numeric(
        split_rows["depth_reduction_pct"], errors="coerce"
    )
    split_rows["total_runtime_seconds"] = pd.to_numeric(
        split_rows["total_runtime_seconds"], errors="coerce"
    )

    single_rows = split_rows[
        ~split_rows["chain_name"].str.contains("__", regex=False)
    ].copy()
    chain_rows = split_rows[
        split_rows["chain_name"].str.contains("__", regex=False)
    ].copy()

    reference_map = {
        f"{split_name}_runtime": [
            build_best_by_metric_row(
                single_rows,
                "total_two_qubit_gate_reduction_pct",
                "total_runtime_seconds",
                "runtime_2q_single",
                "2Q single runtime",
            ),
            build_best_by_metric_row(
                chain_rows,
                "total_two_qubit_gate_reduction_pct",
                "total_runtime_seconds",
                "runtime_2q_chain",
                "2Q chain runtime",
            ),
            build_best_by_metric_row(
                single_rows,
                "depth_reduction_pct",
                "total_runtime_seconds",
                "runtime_depth_single",
                "depth single runtime",
            ),
            build_best_by_metric_row(
                chain_rows,
                "depth_reduction_pct",
                "total_runtime_seconds",
                "runtime_depth_chain",
                "depth chain runtime",
            ),
        ],
        f"{split_name}_2q": [
            build_best_by_metric_row(
                single_rows,
                "total_two_qubit_gate_reduction_pct",
                "total_two_qubit_gate_reduction_pct",
                "2q_single",
                "2Q single",
            ),
            build_best_by_metric_row(
                chain_rows,
                "total_two_qubit_gate_reduction_pct",
                "total_two_qubit_gate_reduction_pct",
                "2q_chain",
                "2Q chain",
            ),
            #build_oracle_metric_row(
            #    split_rows,
            #    "original_circuit_id",
            #    "total_two_qubit_gate_reduction_pct",
            #    "2q_oracle",
            #    "2Q",
            #),
        ],
        f"{split_name}_depth": [
            build_best_by_metric_row(
                single_rows,
                "depth_reduction_pct",
                "depth_reduction_pct",
                "depth_single",
                "depth single",
            ),
            build_best_by_metric_row(
                chain_rows,
                "depth_reduction_pct",
                "depth_reduction_pct",
                "depth_chain",
                "depth chain",
            ),
            #build_oracle_metric_row(
            #    split_rows,
            #    "original_circuit_id",
            #    "depth_reduction_pct",
            #    "depth_oracle",
            #    "depth",
            #),
        ],
    }

    return {
        key: [reference for reference in values if reference is not None]
        for key, values in reference_map.items()
    }


METRIC_REFERENCES = {}
for split_name in ("validation", "test"):
    METRIC_REFERENCES.update(load_split_references(split_name))

plot_frame = comparison[[
    "feature_batch_label",
    "validation_runtime",
    "test_runtime",
    "validation_2q",
    "test_2q",
    "validation_depth",
    "test_depth",
]].copy()

fig, axes = plt.subplots(
    3,
    1,
    figsize=(10.5, max(8.0, 0.52 * len(plot_frame) + 5.5)),
    sharey=True,
    facecolor=FIGURE_BACKGROUND_COLOR,
)
axes = np.atleast_1d(axes)

metric_specs = [
    ((f"{EVAL_SPLIT}_runtime",), f"{EVAL_SPLIT}_runtime", f"Runtime (s)", True),
    ((f"{EVAL_SPLIT}_2q",), f"{EVAL_SPLIT}_2q", f" 2Q Reduction (%)", False),
    ((f"{EVAL_SPLIT}_depth",), f"{EVAL_SPLIT}_depth", f"Depth Reduction (%)", False),
]

legend_entries = {}

for ax, (reference_keys, column, title, ascending) in zip(axes, metric_specs):
    ordered = plot_frame.sort_values(column, ascending=ascending).copy()
    sns.barplot(
        data=ordered,
        x=column,
        y="feature_batch_label",
        hue="feature_batch_label",
        palette=BATCH_COLOR_MAP,
        edgecolor=BAR_EDGE_COLOR,
        linewidth=0.6,
        alpha=0.92,
        dodge=False,
        ax=ax,
        legend=False,
    )
    style_evaluation_axes(ax, grid_axis="x")
    ax.set_title(title, fontsize=RAW_PANEL_TITLE_SIZE, pad=8, weight="semibold")
    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.tick_params(axis="x", labelsize=RAW_X_TICK_SIZE)
    ax.tick_params(axis="y", labelsize=RAW_BATCH_NAME_SIZE)
    for container in ax.containers:
        labels = [f"{bar.get_width():.2f}" for bar in container]
        ax.bar_label(container, labels=labels, padding=4, fontsize=RAW_BAR_VALUE_SIZE, color=FIGURE_TEXT_COLOR)
    ax.margins(x=0.08)

#    reference_rows = []
#    for reference_key in reference_keys:
#        reference_rows.extend(METRIC_REFERENCES.get(reference_key, []))

#    for reference in reference_rows:
#        style = REFERENCE_STYLE_MAP[reference["style_key"]]
#        line = ax.axvline(
#            reference["value"],
#            color=style["color"],
#            linestyle=style["linestyle"],
#            linewidth=2,
#            alpha=0.95,
#        )
#        legend_entries.setdefault(reference["label"], line)

if legend_entries:
    fig.legend(
        list(legend_entries.values())[4:],
        list(legend_entries.keys())[4:],
        loc="lower center",
        bbox_to_anchor=(0.5, -0.03),
        ncol=min(4, len(legend_entries)),
        fontsize=RAW_LEGEND_SIZE,
        frameon=True,
    )

fig.suptitle(
    f"Feature-batch Performance Comparison",
    fontsize=RAW_MAIN_TITLE_SIZE,
    y=0.985,
    weight="semibold",
    color=FIGURE_TEXT_COLOR,
)
fig.subplots_adjust(left=0.24, right=0.97, bottom=0.07, top=0.91, hspace=0.38)
save_figure(fig, f"feature_batch_{EVAL_SPLIT}_raw_metric_comparison")
plt.show()
plt.close(fig)


## 5. Normalized comparison heatmap

This heatmap converts the three validation metrics into comparable 0-to-1 scores, where higher is always better.


In [ ]:
heatmap_frame = comparison.set_index("feature_batch_label")[[
    f"{EVAL_SPLIT}_2q_score" if f"{EVAL_SPLIT}_2q_score" in comparison.columns else "validation_2q_score",
    f"{EVAL_SPLIT}_depth_score" if f"{EVAL_SPLIT}_depth_score" in comparison.columns else "validation_depth_score",
    f"{EVAL_SPLIT}_runtime_score_norm" if f"{EVAL_SPLIT}_runtime_score_norm" in comparison.columns else "validation_runtime_score_norm",
    f"{EVAL_SPLIT}_composite_score" if f"{EVAL_SPLIT}_composite_score" in comparison.columns else "validation_composite_score",
]].rename(columns={
    f"{EVAL_SPLIT}_2q_score" if f"{EVAL_SPLIT}_2q_score" in comparison.columns else "validation_2q_score": "2Q score",
    f"{EVAL_SPLIT}_depth_score" if f"{EVAL_SPLIT}_depth_score" in comparison.columns else "validation_depth_score": "Depth score",
    f"{EVAL_SPLIT}_runtime_score_norm" if f"{EVAL_SPLIT}_runtime_score_norm" in comparison.columns else "validation_runtime_score_norm": "Runtime score",
    f"{EVAL_SPLIT}_composite_score" if f"{EVAL_SPLIT}_composite_score" in comparison.columns else "validation_composite_score": "Composite score",
})

fig, ax = plt.subplots(
    figsize=(9.0, max(4.8, 0.52 * len(heatmap_frame) + 2.2)),
    facecolor=FIGURE_BACKGROUND_COLOR,
)
style_evaluation_axes(ax, grid_axis=None)
sns.heatmap(
    heatmap_frame,
    annot=True,
    fmt=".3f",
    cmap=sns.light_palette("#355F78", as_cmap=True),
    linewidths=0.5,
    linecolor=AXES_BACKGROUND_COLOR,
    cbar_kws={"label": "Normalized score"},
    annot_kws={"size": 15},
    ax=ax,
)
style_evaluation_axes(ax, grid_axis=None)
ax.set_title(
    f"Normalized Feature-Batch Scores\n({EVAL_SPLIT.title()}, Seed-Averaged)",
    fontsize=30,
    pad=9,
    weight="semibold",
)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", labelsize=20, rotation=25)
ax.tick_params(axis="y", labelsize=20, rotation=0)
colorbar = ax.collections[0].colorbar
colorbar.ax.tick_params(labelsize=12, colors=FIGURE_TEXT_COLOR)
colorbar.set_label("Normalized score", fontsize=15, color=FIGURE_TEXT_COLOR)
colorbar.outline.set_edgecolor(FIGURE_BORDER_COLOR)
fig.tight_layout(pad=0.8)
save_figure(fig, f"feature_batch_{EVAL_SPLIT}_heatmap")
plt.show()
plt.close(fig)


## 6. Delta vs baseline

This section compares every experimental batch against `original_no_category` when available. If that batch has not been run yet, the section prints a note and skips the plot.


In [ ]:
baseline_name = FEATURE_BATCH_LABELS["original_no_category"]
selected_2q_col = f"{EVAL_SPLIT}_2q"
selected_depth_col = f"{EVAL_SPLIT}_depth"
selected_runtime_col = f"{EVAL_SPLIT}_runtime"

if baseline_name not in set(comparison["feature_batch_label"]):
    print(f"Skipping delta plot because '{baseline_name}' is not present in the filtered runs yet.")
else:
    baseline_row = comparison[comparison["feature_batch_label"] == baseline_name].iloc[0]
    delta_frame = comparison.copy()
    delta_frame = delta_frame[delta_frame["feature_batch_label"] != baseline_name].copy()
    delta_frame[f"delta_{selected_2q_col}"] = delta_frame[selected_2q_col] - baseline_row[selected_2q_col]
    delta_frame[f"delta_{selected_depth_col}"] = delta_frame[selected_depth_col] - baseline_row[selected_depth_col]
    delta_frame[f"delta_{selected_runtime_col}"] = delta_frame[selected_runtime_col] - baseline_row[selected_runtime_col]

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(14.0, max(5.4, 0.52 * len(delta_frame) + 1.8)),
        sharey=True,
        facecolor=FIGURE_BACKGROUND_COLOR,
    )

    delta_specs = [
        (f"delta_{selected_2q_col}", "2Q reduction change (%)", False),
        (f"delta_{selected_depth_col}", "Depth reduction change (%)", False),
        (f"delta_{selected_runtime_col}", "Runtime change (s)", True),
    ]

    for ax, (column, title, ascending) in zip(axes, delta_specs):
        ordered = delta_frame.sort_values(column, ascending=ascending).copy()
        sns.barplot(
            data=ordered,
            x=column,
            y="feature_batch_label",
            hue="feature_batch_label",
            palette=BATCH_COLOR_MAP,
            edgecolor=BAR_EDGE_COLOR,
            linewidth=0.6,
            alpha=0.92,
            dodge=False,
            ax=ax,
            legend=False,
        )
        style_evaluation_axes(ax, grid_axis="x")
        ax.axvline(0.0, color=FIGURE_BORDER_COLOR, linewidth=1.0)
        ax.set_title(title, fontsize=DELTA_PANEL_TITLE_SIZE, pad=8, weight="semibold")
        ax.set_xlabel("")
        ax.tick_params(axis="x", labelsize=DELTA_X_TICK_SIZE)
        ax.tick_params(axis="y", labelsize=DELTA_BATCH_NAME_SIZE)
        ax.set_ylabel("")
        for container in ax.containers:
            labels = [f"{bar.get_width():+.2f}" for bar in container]
            ax.bar_label(container, labels=labels, padding=4, fontsize=DELTA_BAR_VALUE_SIZE, color=FIGURE_TEXT_COLOR)
        ax.margins(x=0.08)

    fig.suptitle(
        f"Feature-Batch Change Relative to Original without Category ({EVAL_SPLIT.title()})",
        fontsize=DELTA_MAIN_TITLE_SIZE,
        y=0.975,
        weight="semibold",
        color=FIGURE_TEXT_COLOR,
    )
    fig.subplots_adjust(left=0.19, right=0.98, bottom=0.11, top=0.88, wspace=0.28)
    save_figure(fig, f"feature_batch_delta_vs_original_no_category_{EVAL_SPLIT}")
    plt.show()
    plt.close(fig)


## 7. Best-By-Category View

This final table highlights which feature batch wins each validation category after averaging across the tested seeds.


In [ ]:
selected_runtime_col = f"{EVAL_SPLIT}_runtime"
selected_2q_col = f"{EVAL_SPLIT}_2q"
selected_depth_col = f"{EVAL_SPLIT}_depth"
selected_composite_col = "validation_composite_score"

winners = pd.DataFrame([
    {
        "category": f"Lowest {EVAL_SPLIT} runtime",
        "feature_batch": comparison.loc[comparison[selected_runtime_col].idxmin(), "feature_batch_label"],
        "value": comparison[selected_runtime_col].min(),
    },
    {
        "category": f"Highest {EVAL_SPLIT} 2Q reduction",
        "feature_batch": comparison.loc[comparison[selected_2q_col].idxmax(), "feature_batch_label"],
        "value": comparison[selected_2q_col].max(),
    },
    {
        "category": f"Highest {EVAL_SPLIT} depth reduction",
        "feature_batch": comparison.loc[comparison[selected_depth_col].idxmax(), "feature_batch_label"],
        "value": comparison[selected_depth_col].max(),
    },
    {
        "category": "Best validation composite score",
        "feature_batch": comparison.loc[comparison[selected_composite_col].idxmax(), "feature_batch_label"],
        "value": comparison[selected_composite_col].max(),
    },
])

display(winners)
if SAVE_FIGURES:
    print(f"Saved figures to: {FIGURE_DIR}")
